# 06 · Export small JSON files for the website
Every number on the website comes from these files.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "data"
WEB = ROOT / "web" / "public" / "data"
WEB.mkdir(parents=True, exist_ok=True)
results = json.loads((OUT / "results.json").read_text())
pairs = pd.read_parquet(OUT / "pairs.parquet")
comp = pd.read_parquet(OUT / "comparable.parquet")
gaps = pd.read_parquet(OUT / "pair_gaps.parquet")

def save(name, obj):
    path = WEB / f"{name}.json"
    path.write_text(json.dumps(obj, separators=(",", ":"), default=lambda x: None if pd.isna(x) else float(x)))
    print(f"{name}.json {path.stat().st_size / 1024:.1f} KB")

### Coverage: how many pairs at each step

In [2]:
data = pairs.groupby("category").agg(matched=("pair_id", "size"), liquid=("liquid", "sum"), sampled=("in_sample", "sum"),
                                     settled_differently=("settled_differently", "sum")).reset_index()
data["tested"] = data["category"].map(gaps.groupby("category").size()).fillna(0).astype(int)
review = pd.read_csv(OUT / "match_review.csv")
data["reviewed"] = data["category"].map(review.groupby("category").size()).fillna(0).astype(int)
data["review_correct"] = data["category"].map(review.groupby("category")["same_question"].sum()).fillna(0).astype(int)
coverage = data
coverage

,category,matched,liquid,sampled,settled_differently,tested,reviewed,review_correct
0,Fed,52,50,50,0,41,25,25
1,MLB,4583,4580,600,3,270,25,25
2,NBA,1413,1252,600,2,497,25,25
3,NFL,114,114,114,0,114,25,25
4,NHL,1394,1252,600,0,535,25,25


### The gap curve

In [3]:
curve = pd.read_parquet(OUT / "gap_curve.parquet")
curve

,time_left,sort_key,points,pairs,median_gap_pp,mean_gap_pp,p75_gap_pp
0,3–6 h,3.000861,983,983,0.0,0.449440,1.0
1,6–12 h,6.001200,1082,634,0.0,0.406423,1.0
2,12–24 h,12.010222,2750,1406,0.5,0.507727,1.0
3,1–2 days,24.010222,5504,1427,0.5,0.805169,1.0
4,2–3 days,48.010222,4953,1381,1.0,1.768766,2.0
5,3–7 days,72.017778,8639,518,1.0,1.367925,1.5
6,over a week,168.062778,18308,181,1.0,1.670720,2.0


### Pairs for the explorer
Up to 12 per category: the most-traded pairs (both platforms' volume added) with at least 8 comparable points.

In [4]:
pts = comp.groupby("pair_id").size()
data = pairs[pairs["pair_id"].isin(pts[pts >= 8].index)].copy()
data["traded"] = data["kalshi_volume"] + data["poly_volume"]
pick = data.sort_values("traded", ascending=False).groupby("category").head(12)
explorer = []
for r in pick.itertuples():
    s = comp[comp["pair_id"] == r.pair_id].sort_values("ts")
    explorer.append({
        "id": int(r.pair_id), "category": r.category, "date": str(r.date),
        "kalshi_question": r.kalshi_question, "poly_question": r.poly_question,
        "outcome": int(r.kalshi_result), "gap_pp": float(gaps.set_index("pair_id").loc[r.pair_id, "gap_pp"]),
        "series": [[int(t), round(float(k), 4), round(float(p), 4)] for t, k, p in zip(s["ts"], s["kalshi"], s["polymarket"])],
    })
print(len(explorer), "pairs for the explorer")

60 pairs for the explorer


### The pair on the opening screen
Chosen by a fixed rule: among explorer pairs with at least 16 points, the one whose gap is closest to the overall median.

In [5]:
median_gap = results["h1"]["value"]
cands = [e for e in explorer if len(e["series"]) >= 16] or explorer
hero = min(cands, key=lambda e: abs(e["gap_pp"] - median_gap))
print(hero["kalshi_question"], "·", hero["gap_pp"])

Ottawa wins · yes = Ottawa · 0.7500000000000007


### Distribution of pair gaps (for a histogram)

In [6]:
edges = [0, 0.5, 1, 1.5, 2, 3, 4, 6, 10, 100]
hist = pd.cut(gaps["gap_pp"], edges, right=False).value_counts().sort_index()
histogram = [{"from": float(i.left), "to": float(i.right), "pairs": int(v)} for i, v in hist.items()]
histogram

[{'from': 0.0, 'to': 0.5, 'pairs': 205},
 {'from': 0.5, 'to': 1.0, 'pairs': 641},
 {'from': 1.0, 'to': 1.5, 'pairs': 440},
 {'from': 1.5, 'to': 2.0, 'pairs': 95},
 {'from': 2.0, 'to': 3.0, 'pairs': 58},
 {'from': 3.0, 'to': 4.0, 'pairs': 9},
 {'from': 4.0, 'to': 6.0, 'pairs': 5},
 {'from': 6.0, 'to': 10.0, 'pairs': 4},
 {'from': 10.0, 'to': 100.0, 'pairs': 0}]

### Save everything

In [7]:
aliases = json.loads((OUT / "team_code_aliases.json").read_text())
save("overview", {"results": results, "coverage": coverage.to_dict("records"), "histogram": histogram,
                  "aliases": aliases, "hero": hero, "window": ["2025-01-01", "2026-10-31"]})
save("curve", curve.drop(columns=["sort_key"]).to_dict("records"))
save("explorer", explorer)
for f in WEB.glob("*.json"):
    assert f.stat().st_size < 1_000_000, f.name
print("all files under 1 MB")

overview.json 6.1 KB
curve.json 1.0 KB
explorer.json 126.8 KB
all files under 1 MB
